In [ ]:
# ============================================================
#  LOGISTIC REGRESSION — Social Network Ads
#  Dataset : Social_Network_Ads.csv
#  Columns : UserID, Gender, Age, EstimatedSalary, Purchased
# ============================================================


# ── 1. IMPORT LIBRARIES ─────────────────────────────────────
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    confusion_matrix,
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    classification_report,
)

print("=" * 55)
print("       LOGISTIC REGRESSION — Social Network Ads")
print("=" * 55)


# ── 2. LOAD DATASET ─────────────────────────────────────────
df = pd.read_csv("Social_Network_Ads.csv")


# ── 3. BASIC EDA ────────────────────────────────────────────
print("\n📌 BASIC EDA")
print("-" * 40)
print("Shape         :", df.shape)
print("Columns       :", df.columns.tolist())
print("\nFirst 5 rows:")
print(df.head())
print("\nData Types:")
print(df.dtypes)
print("\nStatistical Summary:")
print(df.describe())


# ── 4. MISSING VALUES ───────────────────────────────────────
print("\n📌 MISSING VALUES")
print("-" * 40)
print(df.isnull().sum())

if df.isnull().sum().sum() == 0:
    print("✅ No missing values found.")
else:
    df.dropna(inplace=True)
    print(f"⚠️  Missing values dropped. New shape: {df.shape}")


# ── 5. DUPLICATE VALUES ─────────────────────────────────────
print("\n📌 DUPLICATE VALUES")
print("-" * 40)
duplicates = df.duplicated().sum()
print(f"Duplicate rows: {duplicates}")

if duplicates > 0:
    df.drop_duplicates(inplace=True)
    print(f"⚠️  Duplicates dropped. New shape: {df.shape}")
else:
    print("✅ No duplicate rows found.")


# ── 6. FEATURE SELECTION ────────────────────────────────────
print("\n📌 FEATURE SELECTION")
print("-" * 40)

# Encode 'Gender' column if it exists
if "Gender" in df.columns:
    le = LabelEncoder()
    df["Gender"] = le.fit_transform(df["Gender"])   # Male=1, Female=0
    print("Gender encoded → Male:1, Female:0")

# Drop UserID (not useful for prediction)
cols_to_drop = [col for col in ["User ID", "UserID"] if col in df.columns]
if cols_to_drop:
    df.drop(columns=cols_to_drop, inplace=True)
    print(f"Dropped: {cols_to_drop}")

X = df.drop("Purchased", axis=1)
y = df["Purchased"]

print(f"\nFeatures (X) : {X.columns.tolist()}")
print(f"Target  (y) : Purchased")
print(f"Class distribution:\n{y.value_counts()}")


# ── 7. TRAIN-TEST SPLIT ─────────────────────────────────────
print("\n📌 TRAIN-TEST SPLIT")
print("-" * 40)

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f"Training set  : {X_train.shape[0]} samples")
print(f"Test set      : {X_test.shape[0]} samples")


# ── 8. FEATURE SCALING ──────────────────────────────────────
print("\n📌 FEATURE SCALING (StandardScaler)")
print("-" * 40)

scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)   # fit only on train
X_test  = scaler.transform(X_test)        # transform test separately

print("✅ Scaling done — mean≈0, std≈1")


# ── 9. LOGISTIC REGRESSION ──────────────────────────────────
print("\n📌 LOGISTIC REGRESSION — TRAINING")
print("-" * 40)

model = LogisticRegression(random_state=42)
model.fit(X_train, y_train)

print("✅ Model trained successfully.")


# ── 10. PREDICTION ──────────────────────────────────────────
print("\n📌 PREDICTIONS")
print("-" * 40)

y_pred = model.predict(X_test)

print("Actual   :", list(y_test[:10].values))
print("Predicted:", list(y_pred[:10]))


# ── 11. PREDICT_PROBA ───────────────────────────────────────
print("\n📌 PREDICT_PROBA (first 5 samples)")
print("-" * 40)

y_proba = model.predict_proba(X_test)
print(f"{'Sample':<10} {'P(Not Buy)':<15} {'P(Buy)':<10}")
print("-" * 35)
for i in range(5):
    print(f"{i+1:<10} {y_proba[i][0]:<15.4f} {y_proba[i][1]:.4f}")


# ── 12. CONFUSION MATRIX ────────────────────────────────────
print("\n📌 CONFUSION MATRIX")
print("-" * 40)

cm = confusion_matrix(y_test, y_pred)
print(cm)
print(f"\n  TN={cm[0,0]}  FP={cm[0,1]}")
print(f"  FN={cm[1,0]}  TP={cm[1,1]}")


# ── 13. ACCURACY ────────────────────────────────────────────
print("\n📌 EVALUATION METRICS")
print("-" * 40)

accuracy  = accuracy_score(y_test, y_pred)
precision = precision_score(y_test, y_pred)
recall    = recall_score(y_test, y_pred)
f1        = f1_score(y_test, y_pred)
roc_auc   = roc_auc_score(y_test, y_proba[:, 1])

print(f"Accuracy  : {accuracy:.4f}  ({accuracy*100:.2f}%)")
print(f"Precision : {precision:.4f}")
print(f"Recall    : {recall:.4f}")
print(f"F1 Score  : {f1:.4f}")
print(f"ROC-AUC   : {roc_auc:.4f}")


# ── 18. CLASSIFICATION REPORT ───────────────────────────────
print("\n📌 CLASSIFICATION REPORT")
print("-" * 40)
print(classification_report(y_test, y_pred, target_names=["Not Purchased", "Purchased"]))


# ── 19. FEATURE COEFFICIENTS ────────────────────────────────
print("\n📌 FEATURE COEFFICIENTS")
print("-" * 40)

feature_names = df.drop("Purchased", axis=1).columns.tolist()
coeff_df = pd.DataFrame({
    "Feature"    : feature_names,
    "Coefficient": model.coef_[0]
}).sort_values("Coefficient", ascending=False)

print(coeff_df.to_string(index=False))
print("\nIntercept:", round(model.intercept_[0], 4))
print("→ Higher coeff = stronger positive influence on purchase")


# ── 20. NEW CUSTOMER PREDICTION ─────────────────────────────
print("\n📌 NEW CUSTOMER PREDICTION")
print("-" * 40)

# Feature order MUST match X.columns → ['Gender', 'Age', 'EstimatedSalary']
# Gender: Male=1, Female=0
new_customer = pd.DataFrame([[1, 30, 87000]], columns=['Gender', 'Age', 'EstimatedSalary'])
new_customer_scaled = scaler.transform(new_customer)

new_pred  = model.predict(new_customer_scaled)
new_proba = model.predict_proba(new_customer_scaled)

print(f"Customer  : Gender=Male, Age=30, Salary=87,000")
print(f"Prediction: {'✅ Will Purchase' if new_pred[0] == 1 else '❌ Will NOT Purchase'}")
print(f"Confidence: {max(new_proba[0]) * 100:.2f}%")

print("\n" + "=" * 55)
print("              ✅ PIPELINE COMPLETE")
print("=" * 55)


       LOGISTIC REGRESSION — Social Network Ads

📌 BASIC EDA
----------------------------------------
Shape         : (400, 5)
Columns       : ['User ID', 'Gender', 'Age', 'EstimatedSalary', 'Purchased']

First 5 rows:
    User ID  Gender  Age  EstimatedSalary  Purchased
0  15624510    Male   19            19000          0
1  15810944    Male   35            20000          0
2  15668575  Female   26            43000          0
3  15603246  Female   27            57000          0
4  15804002    Male   19            76000          0

Data Types:
User ID             int64
Gender             object
Age                 int64
EstimatedSalary     int64
Purchased           int64
dtype: object

Statistical Summary:
            User ID         Age  EstimatedSalary   Purchased
count  4.000000e+02  400.000000       400.000000  400.000000
mean   1.569154e+07   37.655000     69742.500000    0.357500
std    7.165832e+04   10.482877     34096.960282    0.479864
min    1.556669e+07   18.000000     1500